# Fabula News: generador de video gratuito (voz clonada + avatar + subtítulos)

Recibe **foto del personaje + guion + título + fuente** y devuelve un MP4 vertical 1080x1920 con:
- la voz de tus videos de HeyGen, clonada con **Chatterbox** (open source, uso comercial permitido) a partir de `voz_referencia.wav`;
- la presentadora animada con **SadTalker**;
- subtítulos sincronizados por palabra (**Whisper**) con la palabra clave resaltada en rosa;
- tarjeta de título al inicio y rótulo con la fuente.

**Cómo usarlo cada vez que vayas a producir:**
1. *Entorno de ejecución* → *Cambiar tipo de entorno* → GPU (T4).
2. *Entorno de ejecución* → *Ejecutar todo*. La primera vez tarda ~8 min (descarga modelos y la voz de referencia del repo).
3. Al final aparece un link `https://xxxx.gradio.live`. Cópialo y pégalo en Fabula News → *Configurar video*.
4. Deja esta pestaña abierta mientras produces. El link dura hasta 72 h.

In [ ]:
#@title 1. Instalar todo (una sola vez por sesión, ~6 min)
import os, sys
if not os.path.exists('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker.git /content/SadTalker
%cd /content/SadTalker

# Voz clonada y transcripción (se instalan primero para que fijen torch)
!pip install -q chatterbox-tts faster-whisper
# Dependencias de SadTalker SIN las versiones viejas que ya no existen para este Python
!pip install -q face_alignment imageio imageio-ffmpeg librosa numba resampy pydub kornia tqdm yacs pyyaml joblib scikit-image basicsr facexlib gfpgan av safetensors
!pip install -q gradio==4.44.1
!apt-get -qq install -y ffmpeg fonts-dejavu-core > /dev/null

# Parche conocido: basicsr importa un módulo que torchvision ya quitó
import glob
for f in glob.glob('/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py'):
    s = open(f).read().replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                               'from torchvision.transforms.functional import rgb_to_grayscale')
    open(f, 'w').write(s)

if not os.path.exists('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    !bash scripts/download_models.sh

import chatterbox, faster_whisper, torch
print('Listo. torch', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
#@title 2. Voz de referencia (se descarga sola del repo)
import os, urllib.request
REF = '/content/voz_referencia.wav'
URL = 'https://raw.githubusercontent.com/yoClaveDeFab/Fabula-news/main/colab/voz_referencia.wav'
if not os.path.exists(REF):
    urllib.request.urlretrieve(URL, REF)
print('Referencia lista:', REF, os.path.getsize(REF)//1024, 'KB')

In [ ]:
#@title 3. Cargar modelos de voz y transcripción
import torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from faster_whisper import WhisperModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = ChatterboxMultilingualTTS.from_pretrained(device=device)
whisper = WhisperModel('small', device=device, compute_type='float16' if device=='cuda' else 'int8')
print('Modelos de voz listos en', device)

In [ ]:
#@title 4. Funciones: voz, subtítulos, avatar y montaje 9:16
import subprocess, glob, os, shutil, time, re, textwrap, html

FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
ROSA = '&HB300FF&'   # formato ASS (BGR): rosa fucsia

def clonar_voz(texto, out_wav):
    # Chatterbox genera por fragmentos; partimos por oraciones para que no se corte
    frases = [f.strip() for f in re.split(r'(?<=[.!?])\s+', texto) if f.strip()]
    partes = []
    for f in frases:
        wav = tts.generate(f, language_id='es', audio_prompt_path=REF, exaggeration=0.45, cfg_weight=0.5)
        partes.append(wav)
        partes.append(torch.zeros(1, int(tts.sr * 0.25)))  # pausa breve entre oraciones
    audio = torch.cat(partes, dim=1)
    torchaudio.save(out_wav, audio, tts.sr)
    return out_wav

def ass_time(s):
    cs = int(round(s * 100))
    return f"{cs//360000}:{(cs%360000)//6000:02d}:{(cs%6000)//100:02d}.{cs%100:02d}"

def subtitulos_ass(wav, out_ass, por_frase=4):
    segs, _ = whisper.transcribe(wav, language='es', word_timestamps=True)
    words = [w for s in segs for w in s.words]
    header = (
        "[Script Info]\nScriptType: v4.00+\nPlayResX: 1080\nPlayResY: 1920\nWrapStyle: 2\n\n"
        "[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        "Style: Fabula,DejaVu Sans,60,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,3,16,0,2,60,60,330,1\n\n"
        "[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n")
    lines = []
    for i in range(0, len(words), por_frase):
        g = words[i:i+por_frase]
        start = g[0].start
        end = words[i+por_frase].start if i+por_frase < len(words) else g[-1].end + 0.4
        longest = max(range(len(g)), key=lambda k: len(g[k].word.strip()))
        txt = ' '.join((f"{{\\c{ROSA}}}{w.word.strip()}{{\\c&HFFFFFF&}}" if k==longest else w.word.strip()) for k, w in enumerate(g))
        lines.append(f"Dialogue: 0,{ass_time(start)},{ass_time(end)},Fabula,,0,0,0,,{txt}")
    open(out_ass, 'w').write(header + '\n'.join(lines) + '\n')
    return out_ass

def animar_avatar(imagen, wav, out_dir):
    cmd = [sys.executable, 'inference.py', '--driven_audio', wav, '--source_image', imagen,
           '--result_dir', out_dir, '--still', '--preprocess', 'full', '--enhancer', 'gfpgan', '--size', '256']
    p = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/SadTalker')
    if p.returncode != 0:
        raise RuntimeError(p.stderr[-1500:])
    vids = glob.glob(f'{out_dir}/**/*.mp4', recursive=True)
    if not vids:
        raise RuntimeError('SadTalker no generó video; usa una foto de frente con la cara despejada.')
    return vids[-1]

def esc(t):
    return t.replace('\\', '\\\\').replace(':', '\\:').replace("'", "\\'").replace('%', '\\%')

def montar(avatar_mp4, wav, ass, titulo, fuente, out_mp4):
    # 1) tarjeta de título 2.5 s (fondo claro, texto negro grande como en los ejemplos)
    tit = '\n'.join(textwrap.wrap(titulo.upper(), 16)[:4])
    intro = out_mp4.replace('.mp4', '_intro.mp4')
    subprocess.run(['ffmpeg','-v','error','-y','-f','lavfi','-i','color=c=0xF4F4F2:s=1080x1920:d=2.5:r=25',
        '-f','lavfi','-i','anullsrc=r=24000:cl=mono','-t','2.5',
        '-vf', f"drawtext=fontfile={FONT}:text='{esc(tit)}':fontsize=96:fontcolor=0x111111:x=(w-text_w)/2:y=(h-text_h)/2:line_spacing=12,"
               f"drawtext=fontfile={FONT}:text='FABULA NEWS':fontsize=34:fontcolor=0x777777:x=(w-text_w)/2:y=h-220",
        '-c:v','libx264','-pix_fmt','yuv420p','-c:a','aac','-shortest', intro], check=True)
    # 2) cuerpo: avatar escalado a 1080 de ancho sobre fondo desenfocado, rótulo de fuente y subtítulos
    cuerpo = out_mp4.replace('.mp4', '_cuerpo.mp4')
    vf = (f"[0:v]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,boxblur=30[bg];"
          f"[0:v]scale=1080:-2[fg];[bg][fg]overlay=(W-w)/2:(H-h)/2-120,"
          f"drawbox=x=60:y=1500:w=960:h=80:color=0x111111@0.85:t=fill,"
          f"drawtext=fontfile={FONT}:text='FUENTE\\: {esc(fuente.upper())}':fontsize=30:fontcolor=white:x=90:y=1524,"
          f"ass={ass}")
    subprocess.run(['ffmpeg','-v','error','-y','-i', avatar_mp4, '-i', wav, '-filter_complex', vf,
        '-map','0:v','-map','1:a','-c:v','libx264','-pix_fmt','yuv420p','-r','25','-c:a','aac','-shortest', cuerpo], check=True)
    # 3) concatenar
    lst = out_mp4.replace('.mp4', '_list.txt')
    open(lst,'w').write(f"file '{intro}'\nfile '{cuerpo}'\n")
    subprocess.run(['ffmpeg','-v','error','-y','-f','concat','-safe','0','-i', lst, '-c','copy', out_mp4], check=True)
    return out_mp4
print('Funciones listas')

In [ ]:
#@title 5. Levantar el servidor y obtener el link para la app
import gradio as gr

def generar(imagen, guion, titulo, fuente, progress=gr.Progress()):
    if imagen is None or not (guion or '').strip():
        raise gr.Error('Faltan la imagen o el guion.')
    work = f'/content/out_{int(time.time())}'
    os.makedirs(work, exist_ok=True)
    try:
        progress(0.1, desc='Clonando voz'); wav = clonar_voz(guion.strip(), f'{work}/voz.wav')
        progress(0.3, desc='Subtítulos');   ass = subtitulos_ass(wav, f'{work}/subs.ass')
        progress(0.4, desc='Animando avatar'); av = animar_avatar(imagen, wav, f'{work}/sad')
        progress(0.9, desc='Montaje 9:16'); final = montar(av, wav, ass, titulo or 'Fabula News', fuente or '', f'{work}/fabula.mp4')
    except Exception as e:
        raise gr.Error(str(e)[-1500:])
    return final

demo = gr.Interface(
    fn=generar,
    inputs=[gr.Image(type='filepath', label='Imagen del personaje'),
            gr.Textbox(label='Guion', lines=8),
            gr.Textbox(label='Título'),
            gr.Textbox(label='Fuente')],
    outputs=gr.Video(label='Video'),
    title='Fabula News video server',
    api_name='generar',
)
demo.queue().launch(share=True)

### Si algo falla
- **No hay GPU disponible**: espera un rato y vuelve a intentar; Colab gratis a veces no tiene T4.
- **Muy lento**: en la celda 4 quita `'--enhancer', 'gfpgan',`. Un guion de 45 s tarda entre 4 y 10 min en T4.
- **La voz no se parece**: cambia `voz_referencia.wav` por otro fragmento de 10 a 30 s donde la voz esté sola, sin música ni efectos. Sube `exaggeration` a 0.6 si suena plana.
- **Error de versión de torch** en SadTalker: en la celda 1, antes de `pip install -r`, agrega `!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121`.
- Las tarjetas intermedias (esquemas, listas) de HeyGen todavía no se generan; es el siguiente paso.